# 3DGS Colab 示例：先跑通 truck，再换自己的室内照片

顺序运行全部 cell。运行时类型必须是 **GPU**，否则 CUDA 一步就会失败。

引擎是官方 `graphdeco-inria/gaussian-splatting`，本 notebook 只做封装。

In [ ]:
import torch, subprocess
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
!nvidia-smi

In [ ]:
# 克隆本仓库（如果 Colab 里还没有 scripts）和官方引擎
import os
if not os.path.exists('/content/3dgs-colab/scripts/train.py'):
    !git clone https://github.com/miaopasi677/3dgs-colab.git /content/3dgs-colab
if not os.path.exists('/content/gaussian-splatting/train.py'):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/gaussian-splatting
print('repo ok:', os.path.exists('/content/3dgs-colab/scripts/train.py'))
print('engine ok:', os.path.exists('/content/gaussian-splatting/train.py'))

In [ ]:
%cd /content/gaussian-splatting
!pip install -q submodules/diff-gaussian-rasterization submodules/simple-knn submodules/fused-ssim
%cd /content/3dgs-colab
print('engine extensions installed')

In [ ]:
# 下载官方示例：Tanks and Temples truck，自带 COLMAP，不用先跑 COLMAP
!python scripts/download_example.py --data-dir data
!ls data/tandt/truck/images | head
!ls data/tandt/truck/sparse/0/

In [ ]:
# 训练：standard = 30000 步。7000 步只是预览，不要用它判断效果。
# T4 上 30000 步需要较长时间，期间不要断开 Colab。
!python scripts/train.py --scene data/tandt/truck --preset standard --engine-dir /content/gaussian-splatting

In [ ]:
# 渲染并找渲染图：官方 render.py 会把图写到 train/ours_30000/renders 下
!python /content/gaussian-splatting/render.py -m output/truck
import pathlib
renders = sorted(pathlib.Path('output/truck').rglob('renders'))
print(renders)
for p in renders:
    print(p, len(list(p.glob('*.png'))))

In [ ]:
# 展示几张渲染图，和原图肉眼对比
from IPython.display import display, Image
import pathlib, glob
files = sorted(glob.glob('output/truck/**/renders/*.png', recursive=True))
print('render files:', len(files))
for f in files[:4]:
    display(Image(f, width=640))

In [ ]:
# 打包下载：最重要就是这个 .ply，拖进 SuperSplat 就能旋转查看
import pathlib, shutil
ply = pathlib.Path('output/truck/point_cloud/iteration_30000/point_cloud.ply')
print('ply exists:', ply.exists(), 'size MB:', ply.stat().st_size/1e6 if ply.exists() else 0)
print('下载方式：左侧文件浏览器里右键 output/truck/point_cloud/iteration_30000/point_cloud.ply → 下载')

## 下一步：换自己的室内照片

1. 把照片上传到 `data/my_room/input/`（固定焦距、重叠 70-80%）
2. 在 Colab 里装 colmap 后跑 `bash scripts/run_colmap.sh data/my_room`
3. 检查 `data/my_room/sparse/0/` 里有 cameras/images/points3D 三个 bin 文件
4. 跑 `python scripts/train.py --scene data/my_room --preset standard --engine-dir /content/gaussian-splatting`

先跑 `fast` 确认流程，再跑 `standard` 判断效果。